# Lab 0 — First contact

**~30 minutes.** Adapted from Week 1 of Ed Donner's *LLM Engineering* course, rebuilt on a
free stack: no credit card, no OpenAI billing.

By the end you will have:

1. a working environment and a call to a hosted model,
2. optionally the same call against a model running on your own laptop,
3. a feel for tokens, temperature, and streaming — the three things that explain most
   surprising LLM behaviour.

**Before you start:** copy `.env.example` to `.env` and paste in your OpenRouter key
(free, no card: <https://openrouter.ai> → Keys).

In [ ]:
# One-time install, if you have not already run pip install -r requirements.txt
# !pip install -q -r ../requirements.txt

In [ ]:
from shared import preflight, ask, chat, stream, client, MODEL

preflight()   # tells you exactly what this machine can reach

### Which free models exist today?

The free tier rotates: models appear and are retired without notice, and a lab that worked
last month can fail with a 404 on the model id. This cell asks OpenRouter what is currently
free **and** lists tool calling as a supported parameter.

That catalog flag is necessary but not sufficient — some ids 403 unless you are inside a
coding agent, and some list "tools" then answer in prose. The **next** cell actually fires
a tool call. Run both before the workshop.

If your `.env` model is not in the list, or the next cell prints `NO TOOL CALL`, pick
another id and restart the kernel.

In [ ]:
import requests

try:
    models = requests.get("https://openrouter.ai/api/v1/models", timeout=30).json()["data"]
    free_with_tools = [m for m in models
                       if m["id"].endswith(":free")
                       and "tools" in (m.get("supported_parameters") or [])]
    print(f"{len(free_with_tools)} free models with tool support:\n")
    for m in sorted(free_with_tools, key=lambda m: m["id"]):
        print(f"  {m['id']:<48} context {m.get('context_length', 0):>9,}")
    print(f"\nYour .env is set to: {MODEL}")
    print("STILL AVAILABLE" if any(m["id"] == MODEL for m in models)
          else "NOT IN THE LIST -> edit .env and pick one above")
except Exception as exc:
    print(f"could not reach the model list ({type(exc).__name__}) — "
          "check https://openrouter.ai/models?q=free in a browser instead")

### Does this model actually call tools?

The list above is a catalog. This cell is a behaviour check: one dummy tool, one question
that requires it. If you see `TOOL CALL OK`, Labs 3, 5 and 6 can use this model. If you see
`NO TOOL CALL` or a 403, edit `MODEL` in `.env` and restart the kernel.

In [ ]:
import json

probe = [{
    "type": "function",
    "function": {
        "name": "get_time_zone",
        "description": "Return the IANA time zone for a city, e.g. Europe/Berlin.",
        "parameters": {
            "type": "object",
            "properties": {"city": {"type": "string"}},
            "required": ["city"],
        },
    },
}]

try:
    reply = chat(
        "What time zone is Berlin in? You must call get_time_zone; do not answer from memory.",
        system="Always call the provided tool. Never answer from memory.",
        tools=probe,
        temperature=0,
    )
    if not reply.tool_calls:
        print("NO TOOL CALL — this model answered in text instead:")
        print((reply.content or "")[:400])
        print("\nLabs 3, 5 and 6 will wobble. Edit MODEL in .env, pick another id from "
              "the list above, restart the kernel, and re-run this cell.")
    else:
        call = reply.tool_calls[0]
        try:
            args = json.loads(call.function.arguments or "{}")
        except json.JSONDecodeError:
            print(f"MALFORMED arguments: {call.function.arguments!r}")
            print("Same fix: pick a different free model.")
        else:
            print(f"TOOL CALL OK: {call.function.name}({args})")
            print("Labs 3, 5 and 6 can use this model.")
except Exception as exc:
    print(f"tool-call probe failed ({type(exc).__name__}: {exc})")
    print("If this is a 403 about 'agentic harnesses', that model is listed as free+tools "
          "but will not serve chat completions — pick another id.")


## 1. The API is smaller than you think

A chat call is: a list of messages, a model name, and a temperature. That is the whole
interface. `shared.py` wraps it in ~10 lines — open the file, it is worth two minutes.

The **system** message sets persona, scope and rules. The **user** message carries the
request. Everything you will build in the next six labs is a way of deciding what goes
into those two strings.

In [ ]:
system = (
    "You explain engineering concepts to sceptical senior backend engineers. "
    "Be concrete, avoid hype words, and never use bullet points."
)
prompt = "In exactly two sentences, what is an AI agent and how is it different from a chatbot?"

print(ask(prompt, system=system))

### Same call, your own laptop

If you installed Ollama (`ollama pull llama3.2`), the identical code runs locally — same
API, different base URL. Compare the answers: the local 3B model is meaningfully worse, and
that gap is the entire argument for hosted frontier models.

In [ ]:
try:
    print(ask(prompt, system=system, local=True))
except Exception as exc:
    print(f"local model unavailable ({type(exc).__name__}) — fine if you skipped Ollama")

## 2. Tokens

Tokens are the unit of price, of latency, and of the context limit. They are also why the
model cannot count the letters in "strawberry" — it never sees letters.

In [ ]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")   # close enough for intuition across models

samples = {
    "english":  "The deployment failed because the database connection pool was exhausted.",
    "json":     '{"status": "failed", "reason": "connection_pool_exhausted", "retries": 3}',
    "code":     "def retry(fn, n=3):\n    for i in range(n):\n        try:\n            return fn()\n        except Exception:\n            continue",
    "russian":  "Развёртывание не удалось: пул соединений с базой данных исчерпан.",
}
for name, text in samples.items():
    n = len(enc.encode(text))
    print(f"{name:8} {len(text):4d} chars  {n:4d} tokens  {len(text)/n:.1f} chars/token")

In [ ]:
my_text = """
Postmortem: the nightly ETL job failed for three consecutive runs. The root cause was a
schema change in the upstream vendor feed: they renamed customer_id to customerId without
notice. Our loader silently wrote nulls for six hours before the freshness alert fired.
Remediation: contract tests against the vendor schema, and an alert on null-rate deltas
rather than on row counts alone.
"""

n = len(enc.encode(my_text))
calls_per_day = 10_000
cost = n * calls_per_day * 3 / 1_000_000
print(f"{n} tokens per call")
print(f"{calls_per_day:,} calls/day at $3/M input tokens = ${cost:,.2f}/day  (${cost*365:,.0f}/year)")

## 3. Temperature

Temperature scales how much the sampler respects the model's probabilities. Near 0 it takes
the most likely token every time; at 1.0 it samples broadly. Run the next cell and watch
the difference — this is why you cannot write an exact-match unit test.

In [ ]:
q = "Name one risk of putting an LLM in a support inbox. One short sentence."

for t in (0.0, 1.0):
    print(f"--- temperature {t} ---")
    for i in range(3):
        print(f"  {i+1}. {ask(q, temperature=t).strip()}")

**Note what you saw.** Temperature 0 is *more repeatable*, not reproducible: the same
prompt can still drift across runs, batches, and model updates. Testing strategy in Lab 2.

## 4. Streaming

Generation is sequential, so progress is free to display. Any user-facing feature should
stream; anything batch should not bother.

In [ ]:
for chunk in stream("Explain prompt injection to a security team in one paragraph."):
    print(chunk, end="", flush=True)

## Stretch goals

1. Swap `MODEL` in `.env` for a different free model (<https://openrouter.ai/models?q=free>)
   and re-run section 1. Note which ones ignore your formatting constraint.
2. Time a call with 50 input tokens vs 5,000 input tokens, same output length. Which budget
   dominates latency?
3. Ask a question you know the answer to, about something obscure in your own domain.
   Judge the answer. That is your first, informal eval — Lab 2 makes it a real one.